In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 39 s


In [2]:
# STANDIN B2
# The three Part B models with the settings chosen on validation in B1 and B2
# (parts/10_baseline_models.ipynb). Dropped from the hand-in notebook, where the real B1 and B2 cells come first.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

tree = DecisionTreeClassifier(max_depth=12, random_state=SEED).fit(X_train, y_train)
logreg = Pipeline([("scale", StandardScaler()),
                   ("model", LogisticRegression(C=10.0, max_iter=2000, random_state=SEED))]).fit(X_train, y_train)
boost = make_boost().fit(X_train, y_train)
MODELS = {"tree": tree, "logreg": logreg, "boosting": boost}

<!-- STEP E1 -->
## Part E: robustness to measurement noise

Everything so far tested the models on clean data. A real sensor measures slightly wrong: clocks drift,
packets are counted twice or missed. **How much measurement error can the models take before they stop
working?**

**How.** We add random **Gaussian noise** to every continuous feature of the **test** set. The size of
the noise is a share (the *level*) of that feature's **training** standard deviation: at level 0.10, a
feature with a standard deviation of 1,000 gets errors of typically ±100. The models are not retrained;
they see noisy versions of flows they were trained to judge on clean data.

- Levels: 0 (clean), 0.05, 0.10, 0.20, 0.50 and 1.00.
- The two 0/1 columns (`Fwd PSH Flags`, `RST Flag Count`) are left alone: a flag is either set or not.
- Values below 0 are set to 0: none of our features is ever negative (packet counts, sizes, times).
- The same noise (seed 42) is used for all three models at a given level, so they are compared on
  identical flows.

Step E3 reads the results: what broke first, and at which level we would stop trusting the model.

In [3]:
# STEP E1
def add_noise(X, level, seed=SEED):
    """Copy of X with Gaussian noise of width level * (training std) on every continuous column.

    The 0/1 columns are not touched, and values are clipped at 0 (no feature is ever negative).
    """
    noisy = X.copy()
    if level == 0:
        return noisy
    rng = np.random.default_rng(seed)
    values = noisy[CONT_COLS].to_numpy(dtype=float, copy=True)   # pandas 3 arrays are read-only
    values += rng.standard_normal(values.shape) * (level * TRAIN_STD[CONT_COLS].to_numpy())
    noisy[CONT_COLS] = np.maximum(values, 0.0)
    return noisy


# Checks: level 0 changes nothing, flags are untouched, nothing becomes negative.
_check = add_noise(X_test.iloc[:2000], 0.5)
assert add_noise(X_test.iloc[:2000], 0.0).equals(X_test.iloc[:2000])
assert _check[FLAG_COLS].equals(X_test.iloc[:2000][FLAG_COLS])
assert (_check.to_numpy() >= 0).all()

# What a noise level means for the three most important features (D1): the training median and the
# typical size of the noise at levels 0.05 and 0.20.
scale = pd.DataFrame({"training_median": TRAIN_MEDIAN, "training_std": TRAIN_STD,
                      "noise_at_0.05": 0.05 * TRAIN_STD, "noise_at_0.20": 0.20 * TRAIN_STD})
scale.loc[["Fwd Seg Size Min", "FWD Init Win Bytes", "Bwd Packets/s", "Flow Duration"]].round(1)

,training_median,training_std,noise_at_0.05,noise_at_0.20
Fwd Seg Size Min,32.0,8.7,0.4,1.7
FWD Init Win Bytes,5792.0,10661.0,533.0,2132.2
Bwd Packets/s,296.7,2159.0,108.0,431.8
Flow Duration,53666.0,5975445.4,298772.3,1195089.1


In [4]:
# STEP E1
NOISE_LEVELS = [0.0, 0.05, 0.10, 0.20, 0.50, 1.00]
rows = []
for level in NOISE_LEVELS:
    X_noisy = add_noise(X_test, level)
    for name, model in MODELS.items():
        rows.append({"level": level, "model": name, **score(model, X_noisy, y_test)})
noise_results = pd.DataFrame(rows)
noise_results.to_csv(TABLES / "E1_noise.csv", index=False, float_format="%.4f")

for metric in ["macro_f1", "recall", "far", "pr_auc"]:
    print(metric)
    print(noise_results.pivot(index="model", columns="level", values=metric)
          .loc[["boosting", "tree", "logreg"]].round(4).to_string(), "\n")

macro_f1
level       0.00    0.05    0.10    0.20    0.50    1.00
model                                                   
boosting  0.9718  0.6618  0.5878  0.5147  0.4563  0.4497
tree      0.9690  0.4727  0.4626  0.4627  0.4601  0.4682
logreg    0.9608  0.9164  0.8477  0.7619  0.6513  0.5847 

recall
level       0.00    0.05    0.10    0.20    0.50    1.00
model                                                   
boosting  0.9923  0.3035  0.1962  0.1043  0.0422  0.0406
tree      0.9849  0.0835  0.0715  0.0850  0.1227  0.1480
logreg    0.9962  0.8490  0.6794  0.5304  0.4117  0.3684 

far
level       0.00    0.05    0.10    0.20    0.50    1.00
model                                                   
boosting  0.0276  0.0217  0.0220  0.0229  0.0324  0.0471
tree      0.0279  0.0816  0.0832  0.1096  0.1793  0.1968
logreg    0.0413  0.0324  0.0334  0.0553  0.1282  0.2021 

pr_auc
level       0.00    0.05    0.10    0.20    0.50    1.00
model                                                  

In [5]:
# STEP E1
# Which features does the noise hurt? Noise at level 0.05 on only the three TCP-settings features
# (the attacker fingerprint, D1), and on all other continuous features but not those three.
TCP_SETTINGS = ["Fwd Seg Size Min", "FWD Init Win Bytes", "Bwd Init Win Bytes"]


def add_noise_to(X, columns, level, seed=SEED):
    """Like add_noise, but only on the given columns (same noise for a column as in add_noise)."""
    noisy = add_noise(X, level, seed)
    keep = [c for c in CONT_COLS if c not in columns]
    noisy[keep] = X[keep]
    return noisy


rows = []
for label, cols in [("all continuous features", CONT_COLS), ("only the 3 TCP-settings features", TCP_SETTINGS),
                    ("all except the 3 TCP-settings features", [c for c in CONT_COLS if c not in TCP_SETTINGS])]:
    X_noisy = add_noise_to(X_test, cols, 0.05)
    for name, model in MODELS.items():
        rows.append({"noise at level 0.05 on": label, "model": name, **score(model, X_noisy, y_test)})
noise_where = pd.DataFrame(rows)
noise_where.to_csv(TABLES / "E1_noise_where.csv", index=False, float_format="%.4f")

# How many test values are exactly 0 (any noise moves them off 0)?
zero_share = float((X_test[CONT_COLS] == 0).to_numpy().mean())
print(f"Share of continuous test values that are exactly 0: {zero_share:.1%}")
noise_where.pivot(index="noise at level 0.05 on", columns="model", values="recall").round(4)

Share of continuous test values that are exactly 0: 34.8%


model,boosting,logreg,tree
noise at level 0.05 on,,,
all continuous features,0.3035,0.8490,0.0835
all except the 3 TCP-settings features,0.8208,0.8489,0.7121
only the 3 TCP-settings features,0.4222,0.9956,0.0753


<!-- STEP E1 -->
### What the noise test shows

| Recall (test) at noise level | 0 | 0.05 | 0.10 | 0.20 | 0.50 | 1.00 |
|---|---:|---:|---:|---:|---:|---:|
| Gradient boosting | 0.992 | **0.304** | 0.196 | 0.104 | 0.042 | 0.041 |
| Decision tree | 0.985 | **0.084** | 0.072 | 0.085 | 0.123 | 0.148 |
| Logistic regression | 0.996 | 0.849 | 0.679 | 0.530 | 0.412 | 0.368 |

| Macro-F1 (test) at noise level | 0 | 0.05 | 0.10 | 0.20 | 0.50 | 1.00 |
|---|---:|---:|---:|---:|---:|---:|
| Gradient boosting | 0.972 | 0.662 | 0.588 | 0.515 | 0.456 | 0.450 |
| Decision tree | 0.969 | 0.473 | 0.463 | 0.463 | 0.460 | 0.468 |
| Logistic regression | 0.961 | 0.916 | 0.848 | 0.762 | 0.651 | 0.585 |

**The two tree models collapse at the smallest noise level.** With noise of only 5% of each feature's
standard deviation, gradient boosting catches 30% of the attacks instead of 99%, and the decision tree
8%. Their macro-F1 falls close to the always-benign baseline (0.424). **Logistic regression degrades
gradually**: it still catches 85% of the attacks at level 0.05 and 53% at 0.20, and keeps the best
macro-F1 at every noise level. The ranking of Part B is reversed: the best model on clean data is one of
the most fragile.

**Why: the tree models match exact values.** The diagnostic table above adds noise at level 0.05 to only
some features:

| Recall at level 0.05, noise on | Gradient boosting | Decision tree | Logistic regression |
|---|---:|---:|---:|
| all continuous features | 0.304 | 0.084 | 0.849 |
| **only the 3 TCP-settings features** | **0.422** | **0.075** | 0.996 |
| all features except those 3 | 0.821 | 0.712 | 0.849 |

Noise on just the three TCP-settings features does most of the damage. The fingerprint (D1) is an
*exact* value, for example a window of 16,383 bytes, and the decision tree tests for a narrow interval
around it (16,368.5 to 16,406.5, B1). A measurement error of a few hundred bytes moves the value out of
that interval, and the flow no longer looks like it comes from an attacker machine. The rest of the
features matter too (recall 0.82 and 0.71 with noise everywhere else): 35% of all test values are exactly
0, and the trees split on "is it 0 or more?", so any positive noise flips those answers. Logistic
regression uses smooth weights instead of exact thresholds, so small errors only shift its score a little.

**The models fail by missing attacks, not by raising alarms.** Gradient boosting's FAR stays between
0.02 and 0.05 at every level: noisy attacks look benign to it. The tree's FAR rises to 0.20 at level 1.0.

**What this means.** Noise of 5% of a standard deviation is small; for `FWD Init Win Bytes` it is about
±530 bytes. Real TCP header fields are recorded exactly and would not drift like this, but timing and
rate features (`Flow Duration`, `Bwd Packets/s`) are measured and can be noisy. The result shows how
narrow the tree models' reasoning is: they work only as long as the fingerprint values arrive exactly as
in training. Step E3 compares this with missing features and decides where to stop trusting the model.

<!-- STEP E2 -->
## Part E: robustness to missing features

A sensor can also fail to record a field at all: a counter is not exported, a timer is lost. A common
fix is to fill the gap with an "ordinary" value, the feature's **training median**. **How many missing
fields can the models take?**

**How.** In every **test** flow we pick a share of the 67 features at random and replace them with their
training median. Each flow gets its own random choice (seed 42); the same choice is used for all three
models at a given level. The models are not retrained.

| Level (share of features missing) | 0 | 0.10 | 0.20 | 0.30 | 0.50 |
|---|---:|---:|---:|---:|---:|
| Features replaced per flow | 0 | 7 | 13 | 20 | 34 |

One thing to keep in mind from D4: the training median of the most important feature, `Fwd Seg Size
Min`, is 32, the value of the *benign* machines. When this feature goes missing and is filled with its
median, the flow loses the attacker fingerprint. A second table below therefore splits the attacks by
whether `Fwd Seg Size Min` was among the missing features.

In [6]:
# STEP E2
def missing_mask(n_rows, frac, seed=SEED):
    """(n_rows, n_features) True where a feature is missing: round(frac * 67) random features per row."""
    k = int(round(frac * len(FEATURES)))
    mask = np.zeros((n_rows, len(FEATURES)), dtype=bool)
    if k > 0:
        rng = np.random.default_rng(seed)
        # k random features per row: the positions of the k smallest of a row of random numbers.
        cols = np.argpartition(rng.random((n_rows, len(FEATURES))), k - 1, axis=1)[:, :k]
        mask[np.arange(n_rows)[:, None], cols] = True
    return mask


def add_missing(X, frac, seed=SEED):
    """Copy of X where round(frac * 67) random features per row are replaced by their training median."""
    values = X.to_numpy(dtype=float, copy=True)                 # copy: pandas 3 arrays are read-only
    mask = missing_mask(len(X), frac, seed)
    medians = np.broadcast_to(TRAIN_MEDIAN[FEATURES].to_numpy(), values.shape)
    values[mask] = medians[mask]
    return pd.DataFrame(values, index=X.index, columns=X.columns)


# Checks: level 0 changes nothing; at level 0.10 exactly 7 features per row were chosen.
assert add_missing(X_test.iloc[:2000], 0.0).equals(X_test.iloc[:2000])
assert (missing_mask(2000, 0.10).sum(axis=1) == 7).all()

MISSING_LEVELS = [0.0, 0.10, 0.20, 0.30, 0.50]
rows = []
for frac in MISSING_LEVELS:
    X_missing = add_missing(X_test, frac)
    for name, model in MODELS.items():
        rows.append({"level": frac, "features_missing": int(round(frac * len(FEATURES))), "model": name,
                     **score(model, X_missing, y_test)})
missing_results = pd.DataFrame(rows)
missing_results.to_csv(TABLES / "E2_missing.csv", index=False, float_format="%.4f")

for metric in ["macro_f1", "recall", "far", "pr_auc"]:
    print(metric)
    print(missing_results.pivot(index="model", columns="level", values=metric)
          .loc[["boosting", "tree", "logreg"]].round(4).to_string(), "\n")

macro_f1
level        0.0     0.1     0.2     0.3     0.5
model                                           
boosting  0.9718  0.9080  0.8514  0.7856  0.6543
tree      0.9690  0.9059  0.8540  0.7999  0.7066
logreg    0.9608  0.8853  0.8234  0.7606  0.6500 

recall
level        0.0     0.1     0.2     0.3     0.5
model                                           
boosting  0.9923  0.8092  0.6656  0.5196  0.2832
tree      0.9849  0.8161  0.6919  0.5770  0.3997
logreg    0.9962  0.8238  0.6978  0.5849  0.3933 

far
level        0.0     0.1     0.2     0.3     0.5
model                                           
boosting  0.0276  0.0248  0.0223  0.0196  0.0136
tree      0.0279  0.0301  0.0322  0.0358  0.0391
logreg    0.0413  0.0572  0.0717  0.0899  0.1155 

pr_auc
level        0.0     0.1     0.2     0.3     0.5
model                                           
boosting  0.9814  0.9472  0.9065  0.8491  0.7176
tree      0.9677  0.8097  0.7049  0.6124  0.4972
logreg    0.9527  0.8412  0.7551  0.

In [7]:
# STEP E2
# Recall on the test attacks, split by whether Fwd Seg Size Min was one of the missing features.
SEG = FEATURES.index("Fwd Seg Size Min")
attack_rows = (y_test == 1).to_numpy()
rows = []
for frac in MISSING_LEVELS[1:]:
    mask = missing_mask(len(X_test), frac)
    X_missing = add_missing(X_test, frac)
    seg_missing = mask[:, SEG]
    for name, model in MODELS.items():
        pred = model.predict(X_missing)
        rows.append({"level": frac, "model": name,
                     "attacks_with_seg_missing": int((attack_rows & seg_missing).sum()),
                     "recall_seg_missing": float(pred[attack_rows & seg_missing].mean()),
                     "recall_seg_present": float(pred[attack_rows & ~seg_missing].mean())})
missing_split = pd.DataFrame(rows)
missing_split.to_csv(TABLES / "E2_missing_split.csv", index=False, float_format="%.4f")
missing_split.round(4)

,level,model,attacks_with_seg_missing,recall_seg_missing,recall_seg_present
0,0.1,tree,1667,0.8026,0.8161
1,0.1,logreg,1667,0.1566,0.9011
2,0.1,boosting,1667,0.0000,0.9030
3,0.2,tree,3110,0.6900,0.6905
4,0.2,logreg,3110,0.1752,0.8234
5,0.2,boosting,3110,0.0000,0.8255
6,0.3,tree,4751,0.5748,0.5758
7,0.3,logreg,4751,0.1661,0.7610
8,0.3,boosting,4751,0.0000,0.7381
9,0.5,tree,8132,0.3935,0.4044


<!-- STEP E2 -->
### What the missing-feature test shows

| Test set, share of features missing | 0 | 0.10 | 0.20 | 0.30 | 0.50 |
|---|---:|---:|---:|---:|---:|
| Recall: gradient boosting | 0.992 | 0.809 | 0.666 | 0.520 | 0.283 |
| Recall: decision tree | 0.985 | 0.816 | 0.692 | 0.577 | 0.400 |
| Recall: logistic regression | 0.996 | 0.824 | 0.698 | 0.585 | 0.393 |
| Macro-F1: gradient boosting | 0.972 | 0.908 | 0.851 | 0.786 | 0.654 |
| Macro-F1: decision tree | 0.969 | 0.906 | 0.854 | 0.800 | 0.707 |
| Macro-F1: logistic regression | 0.961 | 0.885 | 0.823 | 0.761 | 0.650 |
| FAR: gradient boosting | 0.028 | 0.025 | 0.022 | 0.020 | 0.014 |
| FAR: decision tree | 0.028 | 0.030 | 0.032 | 0.036 | 0.039 |
| FAR: logistic regression | 0.041 | 0.057 | 0.072 | 0.090 | 0.116 |

**Missing features hurt all three models, at about the same rate.** With 10% of the features missing (7
per flow), every model loses almost a fifth of its detections (recall about 0.81 instead of 0.99). With
half missing, they catch only 28 to 40% of the attacks. Unlike noise (E1), there is no sudden collapse:
the damage grows steadily with the number of missing fields. At the high levels the decision tree holds
up best (macro-F1 0.71 at 50%), gradient boosting worst on recall (0.28).

**The fingerprint again decides.** The split table shows what happens to an attack when
`Fwd Seg Size Min` is one of its missing features and is filled with the median 32, the benign machines'
value:

| Recall on attacks, `Fwd Seg Size Min` | missing | present |
|---|---:|---:|
| Gradient boosting (level 0.10) | **0.000** | 0.903 |
| Decision tree (level 0.10) | 0.803 | 0.816 |
| Logistic regression (level 0.10) | 0.157 | 0.901 |

**Gradient boosting misses every attack whose `Fwd Seg Size Min` went missing**, at every level: the
single replaced value outweighs the 60 or so features that are still there (the same as the deletion
test in D4). Logistic regression misses most of them too. The decision tree does not care, because its
first question is about `FWD Init Win Bytes` (B1), not `Fwd Seg Size Min`. Even with
`Fwd Seg Size Min` present, recall drops (0.90 at level 0.10), largely because `FWD Init Win Bytes` can
go missing as well and its median, 5,792, is also a benign machine's value.

**Missing values push the models towards "benign".** Filling with medians makes a flow look ordinary,
so recall falls while gradient boosting's FAR even goes down. Logistic regression is the exception: its
FAR rises from 0.04 to 0.12, because some medians push its weighted sum towards "attack".

**What this means.** Median filling is a common default, but here it is not neutral: the median of the
most important features is the benign machines' setting, so every missing fingerprint field quietly turns
an attack into "normal traffic". A safer system would mark a flow with missing key fields as "unknown"
instead of filling it in, which is what the belief rule base in Part F can do. Step E3 decides at which
level to stop trusting the model.